# Pilota commodities: previsioni dei foundation model su GPU

Calcola su una GPU di Colab le previsioni dei foundation model del pilota (`pilota/pilota.py`) e le impacchetta
da riportare sul PC, dove si fa la valutazione con i benchmark classici.

1. *Runtime → Cambia tipo di runtime → **A100 GPU*** (serve Colab Pro/Pro+ o unita' di calcolo; va bene anche L4 o T4,
   solo piu' lente).
2. Esegui le celle in ordine. Alla cella 2 carica il file **`pilota_colab.zip`** (e' nella cartella ML TIME SERIES).
3. Alla fine scarica `risultati_pilota_c<contesto>.zip` e scompattalo dentro la cartella ML TIME SERIES sul PC.

Lavori disponibili alla cella 3:
- **Chronos-2 + spot** (predefinito): Chronos-2 sui 15 futures e i 4 foundation model sulle 5 serie spot dell'energia.
  Circa 15-30 minuti su A100.
- **Volatilita' implicita** (predefinito): i 4 foundation model con VIX su tutte le commodities e con OVX/GVZ
  su energia e metalli preziosi, compiti rendimento e volatilita' (152 previsioni). Circa 1-2 ore su A100.
- **Covariate WTI**: i 4 foundation model sul WTI con covariate (Brent, dollaro, OVX, scorte EIA, placebo),
  5 configurazioni x 3 compiti. Circa 30-60 minuti su A100.
- **Pilota completo**: tutti i foundation model sui 15 futures, con valutazione e report (i benchmark classici
  girano sulla CPU di Colab, circa 20 minuti in piu').

## 1. GPU e pacchetti

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q "timestral @ git+https://github.com/mistralai/timestral.git" "timesfm[torch] @ git+https://github.com/google-research/timesfm.git" "chronos-forecasting>=2.0" statsforecast yfinance arch
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NESSUNA - controlla il tipo di runtime")

## 2. Codice e dati del pilota
Carica `pilota_colab.zip` dal computer, oppure indica dove si trova su Google Drive.

In [ ]:
#@title Carica pilota_colab.zip { display-mode: "form" }
sorgente = "computer"  #@param ["computer", "Google Drive"]
percorso_drive = "/content/drive/MyDrive/ML TIME SERIES/pilota_colab.zip"  #@param {type:"string"}

import os, zipfile, shutil
os.chdir("/content")
if sorgente == "computer":
    from google.colab import files
    nome = next(iter(files.upload()))
else:
    from google.colab import drive
    drive.mount("/content/drive")
    nome = shutil.copy(percorso_drive, "/content/pilota_colab.zip")
zipfile.ZipFile(nome).extractall("/content")
print(sorted(os.listdir("/content/pilota")), "-", len(os.listdir("/content/pilota/dati")), "file di dati")

## 3. Calcola le previsioni

In [ ]:
#@title Parametri { display-mode: "form" }
lavoro = "Volatilita' implicita"  #@param ["Volatilita' implicita", "Covariate WTI", "Chronos-2 + spot", "Pilota completo"]
contesto = 4096  #@param {type:"integer"}
batch = 512  #@param {type:"integer"}

import subprocess, sys, time
SPOT = ["WTI spot", "Brent spot", "Gasolio spot", "Benzina spot", "Gas naturale spot"]
TUTTI = ["Timestral-28M", "Timestral-209M", "TimesFM-3", "Chronos-2"]
base = [sys.executable, "pilota/pilota.py", "--contesto", str(contesto), "--batch", str(batch), "--device", "cuda"]
if lavoro == "Volatilita' implicita":
    comandi = [[sys.executable, "pilota/vol_implicita.py", "--solo-previsioni", "--contesto", str(contesto),
                "--batch", str(min(batch, 256)), "--device", "cuda"]]
elif lavoro == "Covariate WTI":
    comandi = [[sys.executable, "pilota/covariate.py", "--solo-previsioni", "--contesto", str(contesto),
                "--batch", str(min(batch, 128)), "--device", "cuda"]]
elif lavoro == "Chronos-2 + spot":
    comandi = [base + ["--solo-previsioni", "--modelli", "Chronos-2"],
               base + ["--solo-previsioni", "--modelli", *TUTTI, "--asset", *SPOT]]
else:
    comandi = [base + ["--modelli", *TUTTI]]

env = {**os.environ, "HF_HUB_DISABLE_SYMLINKS_WARNING": "1", "PYTHONUNBUFFERED": "1"}
t0 = time.time()
for cmd in comandi:
    print(" ".join(cmd), "\n")
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env, cwd="/content") as p:
        for line in p.stdout:
            if "warn" not in line.lower() and "Fetching" not in line and "Loading weights" not in line:
                print(line, end="")
    print(f"\ncodice di uscita {p.returncode}\n")
print(f"Finito in {(time.time() - t0) / 60:.1f} minuti")

## 4. Risultati (solo con il pilota completo)

In [ ]:
import glob
from IPython.display import Markdown, Image, display
cartella = f"pilota/risultati/contesto_{contesto}" if contesto != 4096 else "pilota/risultati"
if os.path.exists(f"{cartella}/report.md"):
    report = open(f"{cartella}/report.md", encoding="utf-8").read()
    display(Markdown(report.split("## Dettaglio per asset")[0]))
    for png in ["punteggi_relativi.png", "heatmap_asset.png", "sottoperiodi.png", "settori.png", "vol_wti.png"]:
        if os.path.exists(f"{cartella}/{png}"):
            display(Image(f"{cartella}/{png}"))
else:
    print("Nessun report: la valutazione si fa sul PC. Previsioni calcolate:",
          len(glob.glob(f"pilota/cache/*_c{contesto}_p1.npy")),
          "- se e' 0, il calcolo della cella 3 e' fallito: guarda l'errore in fondo al suo output")

## 5. Scarica previsioni e risultati
Lo zip contiene `pilota/cache/` (le previsioni grezze) e, se c'e', la cartella dei risultati, con gli stessi percorsi del PC.

In [ ]:
#@title Scarica { display-mode: "form" }
salva_su_drive = False  #@param {type:"boolean"}
cartella_drive = "/content/drive/MyDrive/ML TIME SERIES"  #@param {type:"string"}

import glob
zip_name = f"risultati_pilota_c{contesto}.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(f"pilota/cache/*_c{contesto}_p1.npy") + glob.glob(f"{cartella}/*"):
        if os.path.isfile(f):
            z.write(f)
print(zip_name, round(os.path.getsize(zip_name) / 1e6, 1), "MB")
if salva_su_drive:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(cartella_drive, exist_ok=True)
    shutil.copy(zip_name, cartella_drive)
    print("Salvato in", cartella_drive)
else:
    from google.colab import files
    files.download(zip_name)